In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('PropertyAIrealestateBangladeshdataset.csv')

print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())
print("\nMissing %:\n", (df.isnull().sum() / len(df) * 100).round(2))

Shape: (33701, 24)

Dtypes:
 area                                     float64
building_type                             object
building_nature                           object
image_url                                 object
num_bath_rooms                           float64
num_bed_rooms                            float64
price                                    float64
property_description                      object
property_overview                         object
property_url                              object
purpose                                   object
city                                      object
locality                                  object
address                                   object
id                                        object
relaxation_amenity_count                   int64
security_amenity_count                     int64
maintenance_or_cleaning_amenity_count      int64
social_amenity_count                       int64
expendable_amenity_count                

In [2]:
df_clean = df.drop(columns=['image_url', 'property_url']).copy()
print("Retained columns:", df_clean.columns.tolist())

Retained columns: ['area', 'building_type', 'building_nature', 'num_bath_rooms', 'num_bed_rooms', 'price', 'property_description', 'property_overview', 'purpose', 'city', 'locality', 'address', 'id', 'relaxation_amenity_count', 'security_amenity_count', 'maintenance_or_cleaning_amenity_count', 'social_amenity_count', 'expendable_amenity_count', 'service_staff_amenity_count', 'unclassify_amenity_count', 'division', 'zone']


In [3]:
# Fill city and division with mode
for col in ['city', 'division']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Impute zone using division+city group mode, fall back to city-level mode
zone_mapping = df_clean.groupby(['division', 'city'])['zone'].agg(
    lambda x: x.mode()[0] if not x.mode().empty else None
)

def impute_zone(row):
    if pd.isna(row['zone']):
        key = (row['division'], row['city'])
        if key in zone_mapping and pd.notna(zone_mapping[key]):
            return zone_mapping[key]
        city_mode = df_clean[df_clean['city'] == row['city']]['zone'].mode()
        return city_mode[0] if not city_mode.empty else np.nan
    return row['zone']

df_clean['zone'] = df_clean.apply(impute_zone, axis=1)

# Fill purpose and id with mode
df_clean['purpose'] = df_clean['purpose'].fillna(df_clean['purpose'].mode()[0])
df_clean['id'] = df_clean['id'].fillna(df_clean['id'].mode()[0])

# Impute amenity counts using zonal median, fallback to global median
amenity_cols = [
    'relaxation_amenity_count', 'security_amenity_count',
    'maintenance_or_cleaning_amenity_count', 'social_amenity_count',
    'expendable_amenity_count', 'service_staff_amenity_count',
    'unclassify_amenity_count'
]
for col in amenity_cols:
    zone_median = df_clean.groupby('zone')[col].transform('median')
    df_clean[col] = df_clean[col].fillna(zone_median).fillna(df_clean[col].median())

print("Remaining nulls:\n", df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

Remaining nulls:
 property_description    14672
property_overview       15192
address                  5177
dtype: int64


In [4]:
df_clean['locality'] = df_clean['locality'].fillna(df_clean['locality'].mode()[0])

cols_to_cap = ['price', 'area', 'num_bed_rooms', 'num_bath_rooms']

for col in cols_to_cap:
    global_q1 = df_clean[col].quantile(0.25)
    global_q3 = df_clean[col].quantile(0.75)

    zone_stats = df_clean.groupby('zone')[col].agg(
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75),
        count='count'
    )

    df_clean['_q1'] = df_clean['zone'].map(zone_stats['q1'])
    df_clean['_q3'] = df_clean['zone'].map(zone_stats['q3'])
    df_clean['_count'] = df_clean['zone'].map(zone_stats['count'])

    # Use global stats for zones with too few records (unstable IQR)
    small_zone = df_clean['_count'] < 5
    df_clean.loc[small_zone, '_q1'] = global_q1
    df_clean.loc[small_zone, '_q3'] = global_q3

    iqr = df_clean['_q3'] - df_clean['_q1']
    lower = (df_clean['_q1'] - 1.5 * iqr).clip(lower=0)
    upper = df_clean['_q3'] + 1.5 * iqr

    outliers = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    df_clean[col] = df_clean[col].clip(lower, upper)
    print(f"'{col}': {outliers} outliers capped")

    df_clean.drop(columns=['_q1', '_q3', '_count'], inplace=True)

# Cast room counts to int after capping
df_clean['num_bed_rooms'] = df_clean['num_bed_rooms'].round().astype(int)
df_clean['num_bath_rooms'] = df_clean['num_bath_rooms'].round().astype(int)
df_clean['price'] = df_clean['price'].round()

print("\nPost-cap stats:\n", df_clean[cols_to_cap].describe())

'price': 3123 outliers capped
'area': 3166 outliers capped
'num_bed_rooms': 3336 outliers capped
'num_bath_rooms': 908 outliers capped

Post-cap stats:
               price          area  num_bed_rooms  num_bath_rooms
count  3.370100e+04  33701.000000   33701.000000     33701.00000
mean   3.090636e+06   1573.158105       2.300436         1.60203
std    5.118575e+06    969.330444       1.268391         1.58418
min    0.000000e+00      0.000000       0.000000         0.00000
25%    2.500000e+04   1000.000000       2.000000         0.00000
50%    1.690000e+05   1350.000000       3.000000         2.00000
75%    5.000000e+06   2030.000000       3.000000         3.00000
max    1.037250e+08   7050.000000       8.000000         8.00000


In [5]:
print("Zero counts:\n", (df_clean[['num_bed_rooms', 'num_bath_rooms']] == 0).sum())

for col in ['num_bed_rooms', 'num_bath_rooms']:
    mask = (df_clean['building_nature'] == 'Residential') & (df_clean[col] == 0)
    if mask.any():
        # Zone median excluding zeros
        nonzero = df_clean[df_clean[col] > 0]
        zone_med = nonzero.groupby('zone')[col].median()
        global_med = nonzero[col].median()
        df_clean.loc[mask, col] = (
            df_clean.loc[mask, 'zone'].map(zone_med).fillna(global_med)
        )
        df_clean[col] = df_clean[col].round().astype(int)
        print(f"Imputed {mask.sum()} zero values in '{col}'")

print("\nZero counts after fix:\n", (df_clean[['num_bed_rooms', 'num_bath_rooms']] == 0).sum())

Zero counts:
 num_bed_rooms      6022
num_bath_rooms    14281
dtype: int64
Imputed 877 zero values in 'num_bed_rooms'
Imputed 8643 zero values in 'num_bath_rooms'

Zero counts after fix:
 num_bed_rooms     5145
num_bath_rooms    5638
dtype: int64


In [6]:
# Strip whitespace from all string columns
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# Restore proper NaN (pandas converts NaN to string "nan" during .astype(str))
df_clean.replace('nan', np.nan, inplace=True)

# Title case for categorical fields
cat_cols = ['building_type', 'building_nature', 'purpose', 'city', 'locality', 'division', 'zone']
for col in cat_cols:
    df_clean[col] = df_clean[col].str.title()

# Enforce MySQL-compatible numeric types
numeric_map = {
    'price': 'float64',
    'area': 'float64',
    'num_bed_rooms': 'int64',
    'num_bath_rooms': 'int64',
    'relaxation_amenity_count': 'int64',
    'security_amenity_count': 'int64',
    'maintenance_or_cleaning_amenity_count': 'int64',
    'social_amenity_count': 'int64',
    'expendable_amenity_count': 'int64',
    'service_staff_amenity_count': 'int64',
    'unclassify_amenity_count': 'int64',
}
df_clean = df_clean.astype(numeric_map)

# Logical column order for MySQL schema
col_order = [
    'id', 'purpose', 'building_nature', 'building_type',
    'division', 'city', 'zone', 'locality', 'address',
    'price', 'area', 'num_bed_rooms', 'num_bath_rooms',
    'relaxation_amenity_count', 'security_amenity_count',
    'maintenance_or_cleaning_amenity_count', 'social_amenity_count',
    'expendable_amenity_count', 'service_staff_amenity_count',
    'unclassify_amenity_count',
    'property_description', 'property_overview'
]
df_clean = df_clean[col_order]

print("Final shape:", df_clean.shape)
print("\nFinal dtypes:\n", df_clean.dtypes)
print("\nSample:\n", df_clean.head())

Final shape: (33701, 22)

Final dtypes:
 id                                        object
purpose                                   object
building_nature                           object
building_type                             object
division                                  object
city                                      object
zone                                      object
locality                                  object
address                                   object
price                                    float64
area                                     float64
num_bed_rooms                              int64
num_bath_rooms                             int64
relaxation_amenity_count                   int64
security_amenity_count                     int64
maintenance_or_cleaning_amenity_count      int64
social_amenity_count                       int64
expendable_amenity_count                   int64
service_staff_amenity_count                int64
unclassify_amenity_count    

In [7]:
df_clean.to_csv('cleaned_properties_mysql_ready.csv', index=False)
print("Saved: cleaned_properties_mysql_ready.csv")

Saved: cleaned_properties_mysql_ready.csv


In [8]:
# Export with explicit NULL as empty string (MySQL LOAD DATA default behavior)
df_clean.to_csv('cleaned_properties_mysql_ready.csv', index=False, na_rep='\\N')
print("CSV saved: cleaned_properties_mysql_ready.csv")
print(f"Rows: {len(df_clean):,} | Columns: {df_clean.shape[1]}")

CSV saved: cleaned_properties_mysql_ready.csv
Rows: 33,701 | Columns: 22


In [9]:
def escape_sql_string(val):
    if pd.isna(val):
        return 'NULL'
    return "'" + str(val).replace("\\", "\\\\").replace("'", "\\'").replace("\n", "\\n").replace("\r", "\\r") + "'"

def to_sql_value(val, dtype):
    if pd.isna(val):
        return 'NULL'
    if pd.api.types.is_integer_dtype(dtype):
        return str(int(val))
    if pd.api.types.is_float_dtype(dtype):
        return str(round(float(val), 2))
    return escape_sql_string(val)

sql_lines = []

sql_lines.append("-- Generated by preprocessing pipeline")
sql_lines.append("-- Import via phpMyAdmin or: mysql -u root -p < properties.sql\n")

sql_lines.append("CREATE DATABASE IF NOT EXISTS `properties_db` DEFAULT CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;")
sql_lines.append("USE `properties_db`;\n")

sql_lines.append("DROP TABLE IF EXISTS `properties`;\n")

sql_lines.append("""CREATE TABLE `properties` (
  `id`                                    VARCHAR(100),
  `purpose`                               VARCHAR(100),
  `building_nature`                       VARCHAR(100),
  `building_type`                         VARCHAR(150),
  `division`                              VARCHAR(100),
  `city`                                  VARCHAR(100),
  `zone`                                  VARCHAR(150),
  `locality`                              VARCHAR(255),
  `address`                               TEXT,
  `price`                                 DOUBLE,
  `area`                                  DOUBLE,
  `num_bed_rooms`                         INT,
  `num_bath_rooms`                        INT,
  `relaxation_amenity_count`              INT,
  `security_amenity_count`               INT,
  `maintenance_or_cleaning_amenity_count` INT,
  `social_amenity_count`                  INT,
  `expendable_amenity_count`              INT,
  `service_staff_amenity_count`           INT,
  `unclassify_amenity_count`              INT,
  `property_description`                  LONGTEXT,
  `property_overview`                     LONGTEXT
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;\n""")

# Batch INSERT for performance (500 rows per statement)
BATCH_SIZE = 500
cols = df_clean.columns.tolist()
col_list = ", ".join(f"`{c}`" for c in cols)

for batch_start in range(0, len(df_clean), BATCH_SIZE):
    batch = df_clean.iloc[batch_start:batch_start + BATCH_SIZE]
    rows = []
    for _, row in batch.iterrows():
        vals = ", ".join(to_sql_value(row[col], df_clean[col].dtype) for col in cols)
        rows.append(f"  ({vals})")
    sql_lines.append(f"INSERT INTO `properties` ({col_list}) VALUES")
    sql_lines.append(",\n".join(rows) + ";\n")

sql_output = "\n".join(sql_lines)

with open('properties.sql', 'w', encoding='utf-8') as f:
    f.write(sql_output)

print(f"SQL saved: properties.sql")
print(f"Total rows: {len(df_clean):,} | Batches: {(len(df_clean) // BATCH_SIZE) + 1}")
print(f"File size: {len(sql_output.encode('utf-8')) / 1024 / 1024:.2f} MB")

SQL saved: properties.sql
Total rows: 33,701 | Batches: 68
File size: 17.43 MB
